# 제12장: 운영 및 생명주기 관리

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch12.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy torch

**Model Retirement Decision Framework**

In [ ]:
"""
Model Retirement Decision Framework - Trigger Detection and Decision Support
"""

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from datetime import date, datetime, timedelta
from enum import Enum
import numpy as np

class RetirementTrigger(Enum):
 """Retirement Trigger Type"""
 # Technical
 PERFORMANCE_DEGRADATION = "Performance Degradation"
 MODEL_OBSOLESCENCE = "Model Obsolescence"
 TECH_STACK_EOL = "Tech Stack EOL"
 SECURITY_VULNERABILITY = "Security Vulnerability"

 # Business
 BUSINESS_VALUE_LOSS = "Business Value Loss"
 STRATEGY_CHANGE = "Strategy Change"
 REPLACEMENT_READY = "Replacement Ready"

 # Regulation
 REGULATORY_CHANGE = "Regulatory Change"
 COMPLIANCE_FAILURE = "Compliance Failure"
 ETHICAL_ISSUE = "Ethical Issue"

 # Operational
 MAINTENANCE_COST = "Maintenance Cost"
 INFRASTRUCTURE_CONSTRAINT = "Infrastructure Constraint"
 INTEGRATION_COMPLEXITY = "Integration Complexity"

class RetirementUrgency(Enum):
 """Retirement Urgency"""
 IMMEDIATE = "Immediate Retirement" # 24 URGENT = "Urgent Retirement" # 1 PLANNED = "Planned Retirement" # 1-3months
 SCHEDULED = "Scheduled Retirement" # 3months

class RetirementDecision(Enum):
 """Retirement Decision"""
 RETIRE = "Retire"
 REMEDIATE = "Remediate and Maintain"
 MONITOR = "Continue Monitoring"
 DEFER = "Defer Decision"

@dataclass
class ModelHealthMetrics:
 """Model Health Metrics"""
 model_id: str
 assessment_date: date

 # Performance Metric
 current_accuracy: float = 0.0
 baseline_accuracy: float = 0.0
 accuracy_trend: str = "stable" # improving, stable, declining
 drift_score: float = 0.0 # 0-1, 1 # Business Metric
 daily_predictions: int = 0
 usage_trend: str = "stable"
 estimated_monthly_value: float = 0.0
 estimated_monthly_cost: float = 0.0

 # Operational Metric
 incidents_last_90days: int = 0
 avg_latency_ms: float = 0.0
 uptime_percent: float = 99.9
 maintenance_hours_monthly: float = 0.0

 # Compliance Metric
 last_audit_date: Optional[date] = None
 audit_issues: int = 0
 documentation_complete: bool = True

 # Technical Metric
 framework_version: str = ""
 framework_eol_date: Optional[date] = None
 security_vulnerabilities: int = 0

@dataclass
class RetirementAssessment:
 """Retire Assessment """
 model_id: str
 assessment_date: date
 assessor: str

 #
 triggered_by: List[RetirementTrigger] = field(default_factory=list)

 #
 technical_score: float = 0.0 # 0-100, business_score: float = 0.0
 compliance_score: float = 0.0
 operational_score: float = 0.0
 overall_score: float = 0.0

 #
 recommended_decision: RetirementDecision = RetirementDecision.MONITOR
 urgency: RetirementUrgency = RetirementUrgency.SCHEDULED
 rationale: str = ""

 # Impact
 affected_systems: List[str] = field(default_factory=list)
 affected_users: int = 0


class ModelRetirementFramework:
 """Model Retirement Decision Framework"""

 # Setup
 THRESHOLDS = {
 'accuracy_drop_critical': 0.15, # Critical: 15% drop from baseline
 'accuracy_drop_warning': 0.10, # Warning: 10% drop
 'drift_critical': 0.3, # Critical: Drift score >= 0.3
 'drift_warning': 0.2,
 'usage_drop_critical': 0.7, # Critical: 70% decrease in usage
 'roi_minimum': 1.0, # Re-evaluate if ROI < 1.0
 'incident_critical': 5, # Critical: >= 5 incidents in 90 days
 'vulnerability_critical': 1, # Security Vulnerability 1 }

 def __init__(self):
 pass

 def assess_retirement(
 self,
 metrics: ModelHealthMetrics,
 assessor: str,
 ) -> RetirementAssessment:
 """Retire Assessment """

 assessment = RetirementAssessment(
 model_id=metrics.model_id,
 assessment_date=date.today(),
 assessor=assessor,
 )

 triggers = []

 # 1. Technical Assessment
 tech_score = 100

 # Performance Degradation
 if metrics.baseline_accuracy > 0:
 accuracy_drop = (metrics.baseline_accuracy - metrics.current_accuracy) / metrics.baseline_accuracy
 if accuracy_drop >= self.THRESHOLDS['accuracy_drop_critical']:
 triggers.append(RetirementTrigger.PERFORMANCE_DEGRADATION)
 tech_score -= 40
 elif accuracy_drop >= self.THRESHOLDS['accuracy_drop_warning']:
 tech_score -= 20

 # Check Drift
 if metrics.drift_score >= self.THRESHOLDS['drift_critical']:
 triggers.append(RetirementTrigger.PERFORMANCE_DEGRADATION)
 tech_score -= 30
 elif metrics.drift_score >= self.THRESHOLDS['drift_warning']:
 tech_score -= 15

 # Check Framework EOL
 if metrics.framework_eol_date:
 days_to_eol = (metrics.framework_eol_date - date.today()).days
 if days_to_eol <= 0:
 triggers.append(RetirementTrigger.TECH_STACK_EOL)
 tech_score -= 30
 elif days_to_eol <= 90:
 tech_score -= 15

 # Security Vulnerability
 if metrics.security_vulnerabilities >= self.THRESHOLDS['vulnerability_critical']:
 triggers.append(RetirementTrigger.SECURITY_VULNERABILITY)
 tech_score -= 40

 assessment.technical_score = max(0, tech_score)

 # 2. Business Assessment
 biz_score = 100

 # Check ROI
 if metrics.estimated_monthly_cost > 0:
 roi = metrics.estimated_monthly_value / metrics.estimated_monthly_cost
 if roi < self.THRESHOLDS['roi_minimum']:
 triggers.append(RetirementTrigger.BUSINESS_VALUE_LOSS)
 biz_score -= 40

 # Check Usage
 if metrics.usage_trend == "declining":
 biz_score -= 20

 assessment.business_score = max(0, biz_score)

 # 3. Compliance Assessment
 compliance_score = 100

 if metrics.audit_issues > 0:
 triggers.append(RetirementTrigger.COMPLIANCE_FAILURE)
 compliance_score -= metrics.audit_issues * 20

 if not metrics.documentation_complete:
 compliance_score -= 20

 assessment.compliance_score = max(0, compliance_score)

 # 4. Operational Assessment
 ops_score = 100

 if metrics.incidents_last_90days >= self.THRESHOLDS['incident_critical']:
 ops_score -= 30

 if metrics.uptime_percent < 99.0:
 ops_score -= 20

 assessment.operational_score = max(0, ops_score)

 # 5. Overall Score and Decision
 assessment.overall_score = (
 assessment.technical_score * 0.3 +
 assessment.business_score * 0.3 +
 assessment.compliance_score * 0.25 +
 assessment.operational_score * 0.15
 )

 assessment.triggered_by = triggers

 # Logic
 assessment.recommended_decision, assessment.urgency = self._determine_decision(
 assessment, triggers
 )

 assessment.rationale = self._generate_rationale(assessment, triggers)

 return assessment

 def _determine_decision(
 self,
 assessment: RetirementAssessment,
 triggers: List[RetirementTrigger],
 ) -> Tuple[RetirementDecision, RetirementUrgency]:
 """Retirement Decision """

 # Immediate Retirement
 immediate_triggers = [
 RetirementTrigger.SECURITY_VULNERABILITY,
 RetirementTrigger.COMPLIANCE_FAILURE,
 RetirementTrigger.ETHICAL_ISSUE,
 ]

 if any(t in triggers for t in immediate_triggers):
 return RetirementDecision.RETIRE, RetirementUrgency.IMMEDIATE

 # Urgent Retirement
 if RetirementTrigger.TECH_STACK_EOL in triggers:
 return RetirementDecision.RETIRE, RetirementUrgency.URGENT

 # Score-based Decision
 if assessment.overall_score < 40:
 return RetirementDecision.RETIRE, RetirementUrgency.PLANNED
 elif assessment.overall_score < 60:
 return RetirementDecision.REMEDIATE, RetirementUrgency.PLANNED
 elif assessment.overall_score < 80:
 return RetirementDecision.MONITOR, RetirementUrgency.SCHEDULED
 else:
 return RetirementDecision.DEFER, RetirementUrgency.SCHEDULED

 def _generate_rationale(
 self,
 assessment: RetirementAssessment,
 triggers: List[RetirementTrigger],
 ) -> str:
 """Generate Decision Rationale"""

 rationale_parts = []

 rationale_parts.append(f"Overall Health Score: {assessment.overall_score:.1f}/100")

 if triggers:
 trigger_names = [t.value for t in triggers]
 rationale_parts.append(f"Active Triggers: {', '.join(trigger_names)}")

 # Weakest Area
 scores = {
 'Technical': assessment.technical_score,
 'Business': assessment.business_score,
 'Compliance': assessment.compliance_score,
 'Operational': assessment.operational_score,
 }
 weakest = min(scores, key=scores.get)
 rationale_parts.append(f"Weakest Area: {weakest} ({scores[weakest]:.1f} )")

 return "; ".join(rationale_parts)

 def generate_retirement_report(self, assessment: RetirementAssessment) -> str:
 """Retire Assessment Report Generation - ( Guide Retention)"""
 # ( : Code days Report Generation Logic)
 return "# Model Retire Assessment Report ( )"

**좀비 모델 탐지 자동화 시스템**

In [ ]:
"""
Zombie Model Detection System
- 사용량, 성능, 비용, 소유자 메트릭을 기반으로 좀비 모델 자동 탐지
- 정기 스캔(주간/월간)으로 레지스트리 내 전체 모델을 점검
"""

from dataclasses import dataclass, field
from datetime import date, timedelta
from typing import List, Dict, Optional
from enum import Enum
import logging

logger = logging.getLogger(__name__)

class ZombieRiskLevel(Enum):
    """좀비 위험 등급"""
    CRITICAL = "Critical"    # 즉시 Retire 권고
    HIGH = "High"            # 30일 내 소유자 확인 필요
    MEDIUM = "Medium"        # 모니터링 강화
    LOW = "Low"              # 정상

@dataclass
class ModelRegistryEntry:
    """모델 레지스트리 항목"""
    model_id: str
    model_name: str
    owner: Optional[str]
    team: Optional[str]
    deployed_date: date
    last_prediction_date: Optional[date]
    framework: str
    framework_version: str
    endpoint_url: Optional[str] = None
    tags: List[str] = field(default_factory=list)

@dataclass
class ModelUsageMetrics:
    """모델 사용량 메트릭"""
    model_id: str
    daily_request_count_avg_30d: float = 0.0
    daily_request_count_avg_90d: float = 0.0
    unique_consumers_30d: int = 0
    last_request_timestamp: Optional[date] = None

@dataclass
class ModelCostMetrics:
    """모델 비용 메트릭"""
    model_id: str
    monthly_compute_cost: float = 0.0
    monthly_storage_cost: float = 0.0
    monthly_network_cost: float = 0.0
    estimated_monthly_value: float = 0.0

    @property
    def total_monthly_cost(self) -> float:
        return (self.monthly_compute_cost
                + self.monthly_storage_cost
                + self.monthly_network_cost)

    @property
    def roi(self) -> float:
        if self.total_monthly_cost == 0:
            return 0.0
        return self.estimated_monthly_value / self.total_monthly_cost

@dataclass
class ZombieDetectionResult:
    """좀비 탐지 결과"""
    model_id: str
    risk_level: ZombieRiskLevel
    reasons: List[str] = field(default_factory=list)
    days_since_last_use: Optional[int] = None
    monthly_waste_cost: float = 0.0
    recommended_action: str = ""

class ZombieModelDetector:
    """
    좀비 모델 탐지기

    탐지 기준:
    1. 사용량 기준: 90일간 평균 요청 수 < threshold
    2. 소유자 기준: owner 필드가 비어 있거나 퇴사자
    3. 비용 기준: ROI < 1.0 (비용 > 가치)
    4. 보안 기준: EOL된 프레임워크 사용
    5. 기간 기준: 마지막 사용일로부터 N일 이상 경과
    """

    DEFAULT_CONFIG = {
        'idle_days_critical': 180,      # 180일 미사용 -> Critical
        'idle_days_warning': 90,        # 90일 미사용 -> High
        'min_daily_requests': 1.0,      # 일 평균 1건 미만 -> 의심
        'min_unique_consumers': 1,      # 소비자 0 -> 의심
        'roi_threshold': 1.0,           # ROI 1.0 미만 -> 비용 초과
        'eol_frameworks': [             # EOL된 프레임워크 목록
            'tensorflow-1.x',
            'pytorch-1.x',
            'scikit-learn-0.x',
        ],
    }

    def __init__(self, config: Optional[Dict] = None):
        self.config = config or self.DEFAULT_CONFIG
        self.departed_employees: List[str] = []  # HR 연동

    def scan_registry(
        self,
        registry: List[ModelRegistryEntry],
        usage_data: Dict[str, ModelUsageMetrics],
        cost_data: Dict[str, ModelCostMetrics],
    ) -> List[ZombieDetectionResult]:
        """전체 레지스트리 스캔"""
        results = []

        for entry in registry:
            result = self._evaluate_model(
                entry,
                usage_data.get(entry.model_id),
                cost_data.get(entry.model_id),
            )
            if result.risk_level != ZombieRiskLevel.LOW:
                results.append(result)
                logger.warning(
                    f"Zombie detected: {entry.model_id} "
                    f"[{result.risk_level.value}] "
                    f"- {', '.join(result.reasons)}"
                )

        # 위험도 순 정렬
        results.sort(
            key=lambda r: list(ZombieRiskLevel).index(r.risk_level)
        )
        return results

    def _evaluate_model(
        self,
        entry: ModelRegistryEntry,
        usage: Optional[ModelUsageMetrics],
        cost: Optional[ModelCostMetrics],
    ) -> ZombieDetectionResult:
        """개별 모델 좀비 여부 평가"""
        reasons = []
        risk_scores = []
        monthly_waste = 0.0
        days_idle = None

        # 1. 사용량 점검
        if usage:
            if usage.last_request_timestamp:
                days_idle = (date.today()
                             - usage.last_request_timestamp).days
                if days_idle >= self.config['idle_days_critical']:
                    reasons.append(
                        f"{days_idle}일간 미사용 (Critical)")
                    risk_scores.append(3)
                elif days_idle >= self.config['idle_days_warning']:
                    reasons.append(
                        f"{days_idle}일간 미사용 (Warning)")
                    risk_scores.append(2)

            if (usage.daily_request_count_avg_90d
                    < self.config['min_daily_requests']):
                reasons.append("일 평균 요청 수 1건 미만")
                risk_scores.append(2)

            if (usage.unique_consumers_30d
                    < self.config['min_unique_consumers']):
                reasons.append("활성 소비자 없음")
                risk_scores.append(2)
        else:
            reasons.append("사용량 데이터 없음")
            risk_scores.append(2)

        # 2. 소유자 점검
        if not entry.owner:
            reasons.append("소유자 미지정")
            risk_scores.append(2)
        elif entry.owner in self.departed_employees:
            reasons.append(
                f"소유자 퇴사: {entry.owner}")
            risk_scores.append(2)

        # 3. 비용 점검
        if cost:
            if cost.roi < self.config['roi_threshold']:
                reasons.append(
                    f"ROI {cost.roi:.2f} (임계값 미만)")
                risk_scores.append(2)
                monthly_waste = (cost.total_monthly_cost
                                 - cost.estimated_monthly_value)

        # 4. 프레임워크 EOL 점검
        fw_key = (f"{entry.framework}-"
                  f"{entry.framework_version.split('.')[0]}.x")
        if fw_key in self.config['eol_frameworks']:
            reasons.append(
                f"EOL 프레임워크: {entry.framework} "
                f"{entry.framework_version}")
            risk_scores.append(2)

        # 종합 위험도 결정
        if not risk_scores:
            risk_level = ZombieRiskLevel.LOW
        elif max(risk_scores) >= 3 or len(risk_scores) >= 3:
            risk_level = ZombieRiskLevel.CRITICAL
        elif max(risk_scores) >= 2:
            risk_level = ZombieRiskLevel.HIGH
        else:
            risk_level = ZombieRiskLevel.MEDIUM

        # 권고 액션 결정
        action_map = {
            ZombieRiskLevel.CRITICAL:
                "즉시 Retire 프로세스 개시",
            ZombieRiskLevel.HIGH:
                "30일 내 소유자 확인 및 Retire 검토",
            ZombieRiskLevel.MEDIUM:
                "모니터링 강화 및 분기 내 재평가",
            ZombieRiskLevel.LOW: "정상 운영",
        }

        return ZombieDetectionResult(
            model_id=entry.model_id,
            risk_level=risk_level,
            reasons=reasons,
            days_since_last_use=days_idle,
            monthly_waste_cost=max(0, monthly_waste),
            recommended_action=action_map[risk_level],
        )

    def generate_summary_report(
        self,
        results: List[ZombieDetectionResult],
    ) -> Dict:
        """좀비 탐지 요약 리포트 생성"""
        summary = {
            'scan_date': date.today().isoformat(),
            'total_scanned': 0,
            'zombies_found': len(results),
            'by_risk_level': {},
            'total_monthly_waste': sum(
                r.monthly_waste_cost for r in results),
            'top_waste_models': [],
        }

        for level in ZombieRiskLevel:
            count = sum(
                1 for r in results
                if r.risk_level == level)
            summary['by_risk_level'][level.value] = count

        # 비용 낭비 상위 5개 모델
        sorted_by_waste = sorted(
            results,
            key=lambda r: r.monthly_waste_cost,
            reverse=True,
        )[:5]
        summary['top_waste_models'] = [
            {
                'model_id': r.model_id,
                'monthly_waste': r.monthly_waste_cost,
                'reasons': r.reasons,
            }
            for r in sorted_by_waste
        ]

        return summary

**Model Retirement Management System Implementation**

In [ ]:
class RetirementPhase(Enum):
 INITIATED = "Initiated"; ASSESSMENT = "Assessment"; APPROVED = "Approved"
 NOTIFICATION = "Notification"; MIGRATION = "Migration"; ARCHIVING = "Archiving"
 SHUTDOWN = "Shutdown"; CLEANUP = "Cleanup"; DOCUMENTED = "Documented"
 COMPLETED = "Completed"

@dataclass
class RetirementProject:
 project_id: str; model_id: str
 current_phase: RetirementPhase = RetirementPhase.INITIATED
 urgency: RetirementUrgency = RetirementUrgency.PLANNED
 tasks: List[RetirementTask] = field(default_factory=list)
 archive_location: str = ""

class ModelRetirementManager:
 """Model Retire Workflow Management """
 PHASE_TASKS = {
 RetirementPhase.ARCHIVING: ["Archive Model Artifacts", "Archive Training Data References", "Archive Documentation"],
 RetirementPhase.SHUTDOWN: ["Deactivate API Endpoints", "Disable Monitoring Alerts"]
 }

 def advance_phase(self, project_id: str) -> RetirementProject:
 # Transition to next phase after checking task completion
 pass

**SISA 기반 머신 언러닝 구현**

In [ ]:
"""
SISA (Sharded, Isolated, Sliced, Aggregated) Machine Unlearning
- 학습 데이터를 Shard로 분할, 각 Shard 내에서 Slice 단위 점진 학습
- 삭제 요청 시 해당 Shard의 구성 모델만 재학습
- Bourtoule et al., 2021 기반 구현
"""

from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable
from datetime import datetime
import hashlib
import numpy as np
import logging

logger = logging.getLogger(__name__)

@dataclass
class UnlearningRequest:
    """데이터 삭제(언러닝) 요청"""
    request_id: str
    data_subject_id: str
    data_point_ids: List[str]
    reason: str  # "GDPR Art.17", "CCPA", "voluntary"
    requested_at: datetime = field(
        default_factory=datetime.now)
    completed_at: Optional[datetime] = None
    status: str = "pending"  # pending, processing, completed

@dataclass
class ShardCheckpoint:
    """Shard별 학습 체크포인트"""
    shard_id: int
    slice_index: int
    model_weights: Any  # 직렬화된 모델 가중치
    training_data_ids: List[str]
    created_at: datetime = field(
        default_factory=datetime.now)
    checksum: str = ""

    def compute_checksum(self) -> str:
        """체크포인트 무결성 해시"""
        content = f"{self.shard_id}:{self.slice_index}"
        content += f":{len(self.training_data_ids)}"
        self.checksum = hashlib.sha256(
            content.encode()).hexdigest()[:16]
        return self.checksum

class SISATrainer:
    """
    SISA 학습 및 언러닝 관리자

    핵심 아이디어:
    - 전체 학습 데이터를 num_shards개 샤드로 분할
    - 각 샤드를 num_slices개 슬라이스로 나누어 점진 학습
    - 각 슬라이스 학습 완료 시 체크포인트 저장
    - 삭제 요청 시, 해당 데이터가 속한 샤드만 재학습
    """

    def __init__(
        self,
        num_shards: int = 5,
        num_slices: int = 10,
        model_factory: Optional[Callable] = None,
        aggregation: str = "majority_vote",
    ):
        self.num_shards = num_shards
        self.num_slices = num_slices
        self.model_factory = model_factory
        self.aggregation = aggregation

        # 샤드별 구성 모델
        self.constituent_models: Dict[int, Any] = {}
        # 데이터 포인트 -> 샤드 매핑
        self.data_to_shard: Dict[str, int] = {}
        # 샤드별 체크포인트 히스토리
        self.checkpoints: Dict[int, List[ShardCheckpoint]] = {
            i: [] for i in range(num_shards)
        }
        # 삭제 요청 이력
        self.unlearn_history: List[UnlearningRequest] = []

    def assign_data_to_shards(
        self,
        data_ids: List[str],
    ) -> Dict[int, List[str]]:
        """데이터를 샤드에 균등 배정 (해시 기반)"""
        shard_assignments: Dict[int, List[str]] = {
            i: [] for i in range(self.num_shards)
        }

        for data_id in data_ids:
            # 결정론적 샤드 배정 (재현 가능)
            hash_val = int(hashlib.md5(
                data_id.encode()).hexdigest(), 16)
            shard_id = hash_val % self.num_shards
            shard_assignments[shard_id].append(data_id)
            self.data_to_shard[data_id] = shard_id

        logger.info(
            f"데이터 {len(data_ids)}건을 "
            f"{self.num_shards}개 샤드에 배정 완료"
        )
        return shard_assignments

    def train_shard(
        self,
        shard_id: int,
        shard_data: List[str],
        from_slice: int = 0,
    ) -> None:
        """
        특정 샤드의 구성 모델 학습
        from_slice > 0이면 해당 슬라이스의 체크포인트부터 재개
        """
        slice_size = max(1, len(shard_data) // self.num_slices)

        # 체크포인트에서 재개
        if from_slice > 0 and self.checkpoints[shard_id]:
            checkpoint = self.checkpoints[shard_id][
                from_slice - 1]
            model = self._load_model(checkpoint.model_weights)
            logger.info(
                f"Shard {shard_id}: "
                f"Slice {from_slice}부터 재개")
        else:
            model = self.model_factory()
            from_slice = 0

        # 슬라이스 단위 점진 학습
        for s in range(from_slice, self.num_slices):
            start_idx = 0
            end_idx = min((s + 1) * slice_size,
                          len(shard_data))
            current_data = shard_data[:end_idx]

            # 모델 학습 (구현체에 위임)
            model = self._train_model(
                model, current_data)

            # 체크포인트 저장
            checkpoint = ShardCheckpoint(
                shard_id=shard_id,
                slice_index=s,
                model_weights=self._serialize_model(model),
                training_data_ids=current_data.copy(),
            )
            checkpoint.compute_checksum()

            # 기존 체크포인트 교체 또는 추가
            if s < len(self.checkpoints[shard_id]):
                self.checkpoints[shard_id][s] = checkpoint
            else:
                self.checkpoints[shard_id].append(
                    checkpoint)

            logger.info(
                f"Shard {shard_id}, Slice {s} "
                f"학습 완료 (데이터: {len(current_data)}건)")

        self.constituent_models[shard_id] = model

    def process_unlearn_request(
        self,
        request: UnlearningRequest,
        full_shard_data: Dict[int, List[str]],
    ) -> Dict:
        """
        언러닝 요청 처리
        1. 삭제 대상 데이터가 속한 샤드 식별
        2. 해당 데이터 제거 후 영향받는 샤드만 재학습
        3. 재학습은 가장 가까운 체크포인트에서 시작
        """
        request.status = "processing"
        affected_shards = set()
        retrain_info = {}

        # 1. 영향받는 샤드 식별
        for data_id in request.data_point_ids:
            shard_id = self.data_to_shard.get(data_id)
            if shard_id is not None:
                affected_shards.add(shard_id)

        logger.info(
            f"언러닝 요청 {request.request_id}: "
            f"{len(request.data_point_ids)}건 삭제, "
            f"{len(affected_shards)}개 샤드 영향")

        # 2. 영향받는 샤드만 재학습
        for shard_id in affected_shards:
            # 삭제 대상 제거
            original_data = full_shard_data[shard_id]
            cleaned_data = [
                d for d in original_data
                if d not in request.data_point_ids
            ]

            # 가장 가까운 안전 체크포인트 탐색
            safe_slice = self._find_safe_checkpoint(
                shard_id, request.data_point_ids)

            # 재학습 실행
            self.train_shard(
                shard_id, cleaned_data,
                from_slice=safe_slice)

            retrain_info[shard_id] = {
                'original_size': len(original_data),
                'cleaned_size': len(cleaned_data),
                'retrain_from_slice': safe_slice,
            }

            # 매핑 정보 업데이트
            for data_id in request.data_point_ids:
                self.data_to_shard.pop(data_id, None)

            full_shard_data[shard_id] = cleaned_data

        # 3. 완료 기록
        request.status = "completed"
        request.completed_at = datetime.now()
        self.unlearn_history.append(request)

        return {
            'request_id': request.request_id,
            'affected_shards': list(affected_shards),
            'retrain_info': retrain_info,
            'total_shards': self.num_shards,
            'unaffected_shards': (
                self.num_shards - len(affected_shards)),
        }

    def _find_safe_checkpoint(
        self,
        shard_id: int,
        removed_ids: List[str],
    ) -> int:
        """삭제 대상이 포함되지 않은 가장 마지막 체크포인트"""
        removed_set = set(removed_ids)

        for i, cp in enumerate(
                self.checkpoints[shard_id]):
            if removed_set & set(cp.training_data_ids):
                return max(0, i)

        return 0

    def _train_model(self, model, data):
        """모델 학습 (구현체에 위임)"""
        # 실제 구현에서는 PyTorch/TensorFlow 학습 루프
        return model

    def _serialize_model(self, model) -> bytes:
        """모델 직렬화"""
        return b""  # 실제: pickle/torch.save

    def _load_model(self, weights: bytes):
        """체크포인트에서 모델 복원"""
        return self.model_factory()

    def get_unlearning_audit_log(self) -> List[Dict]:
        """언러닝 감사 로그 생성 (규제 대응)"""
        return [
            {
                'request_id': r.request_id,
                'data_subject': r.data_subject_id,
                'num_points_removed': len(r.data_point_ids),
                'reason': r.reason,
                'requested_at': r.requested_at.isoformat(),
                'completed_at': (
                    r.completed_at.isoformat()
                    if r.completed_at else None),
                'status': r.status,
            }
            for r in self.unlearn_history
        ]

**Retirement Retrospective and Lessons Learned Management System**

In [ ]:
class LessonCategory(Enum):
 DATA = "Data"; MODELING = "Modeling"; GOVERNANCE = "Governance"

@dataclass
class LessonLearned:
 lesson_id: str; category: LessonCategory; title: str
 description: str; context: str; recommendation: str

class RetirementRetrospectiveManager:
 """Manager of turning mistakes into knowledge"""
 def add_lesson(self, retro_id: str, lesson: LessonLearned):
 # Register lessons in Knowledge Base (KB) for future retrieval
 pass

**Lineage-based Retirement Impact Analyzer**

In [ ]:
class DependencyType(Enum):
 DIRECT_CONSUMER = "Direct Consumer"; ENSEMBLE_MEMBER = "Ensemble Member"
 FALLBACK_MODEL = "Fallback Model"

class RetirementImpactAnalyzer:
 """Cascade failure analysis via lineage graph"""
 def analyze_impact(self, model_id: str, lineage_manager):
 # Call IntegratedLineageManager from Ch 9 for downstream tracking
 downstream = lineage_manager.get_downstream(model_id)
 # Impact Retire (Blocker) Setup
 pass